# LLP hands-on: reconstructing a Heavy Neutral Lepton

This sample is a simulated **Heavy Neutral Lepton (HNL)** — a hypothetical
long-lived particle. Each HNL flies tens of metres downstream and decays into
**two daughters** at a displaced vertex (just the kind of signature SHiP looks for).

You'll use **ROOT's RDataFrame** to:
1. reconstruct the HNL **invariant mass** from its daughters,
2. study how detector **momentum resolution** smears that measurement.

Run the cells top to bottom (Shift+Enter). The **TODO** cells are yours to fill in.

## 0. Setup

Runs on **SWAN** with any recent LCG stack (ROOT + Python included).
`%jsroot on` makes ROOT plots interactive inside the notebook.

In [ ]:
%jsroot on
import ROOT
print("ROOT", ROOT.gROOT.GetVersion())

## 1. The data (on EOS)

The samples live on **EOS** — CERN's shared storage, mounted at `/eos/...` on SWAN
and lxplus. We don't copy data into the git repo; we read it straight from EOS.

The tree is called **`Events`**, one row per HNL decay. The branches we'll use:

| branch | meaning |
|--------|---------|
| `LLP_m` | the *true* HNL mass (what we should reconstruct) |
| `vtx_x, vtx_y, vtx_z` | the decay vertex position [m] |
| `d_px, d_py, d_pz, d_m` | the two daughters' momenta & masses (a vector per event) |

First, locate the file on EOS:

In [ ]:
import os, glob

EOS_DIR = "/eos/user/m/matclim/SHiPsim/SS_2026_data"
named   = f"{EOS_DIR}/HNL_1.000e+00_1.000e+00_0.000e+00_1.000e+00_0.000e+00_data.root"

if os.path.exists(named):
    DATA = named                                   # the HNL sample on EOS
elif glob.glob(f"{EOS_DIR}/*.root"):
    DATA = sorted(glob.glob(f"{EOS_DIR}/*.root"))[0]   # any sample in that folder
else:
    DATA = "data/hnl_data.root"                    # fallback: local copy in the repo

print("reading:", DATA)

In [ ]:
df = ROOT.RDataFrame("Events", DATA)

print("events:    ", df.Count().GetValue())
print("true mass: ", round(df.Mean("LLP_m").GetValue(), 4), "GeV")

The HNL is *long-lived*, so it decays far from the collision point. Let's see where
along the beam (`vtx_z`):

In [ ]:
c0 = ROOT.TCanvas("c0", "", 700, 500)
h_z = df.Histo1D(("h_z", "decay vertex; z [m]; decays", 100, 0, 100), "vtx_z")
h_z.Draw()
c0.Draw()

The decays sit tens of metres downstream — that displaced vertex is the key
experimental handle for finding long-lived particles.

## 2. Reconstruct the invariant mass

The HNL's mass is the invariant mass of its two daughters: add their 4-momenta
and take $m = \sqrt{E^2 - p_x^2 - p_y^2 - p_z^2}$.

In RDataFrame you build new columns with **`Define`** (string expressions, evaluated
in C++). `Sum(...)` adds up the two-element daughter vector.

In [ ]:
reco = (df
    .Filter("d_px.size() == 2", "exactly two daughters")
    .Define("d_E",    "sqrt(d_px*d_px + d_py*d_py + d_pz*d_pz + d_m*d_m)")  # per-daughter energy
    .Define("px_tot", "Sum(d_px)")
    .Define("py_tot", "Sum(d_py)")
    .Define("pz_tot", "Sum(d_pz)")
    .Define("E_tot",  "Sum(d_E)")
    .Define("m_inv",  "sqrt(E_tot*E_tot - px_tot*px_tot - py_tot*py_tot - pz_tot*pz_tot)")
)

c1 = ROOT.TCanvas("c1", "", 700, 500)
h_m = reco.Histo1D(("h_m", "reconstructed mass; m_{inv} [GeV]; events", 60, 0.9, 1.1), "m_inv")
h_m.Draw()
c1.Draw()

A sharp peak at **1 GeV** — you just reconstructed the HNL mass from its decay products.

## 3. TODO — reconstruct the transverse momentum

The HNL's transverse momentum is $p_T = \sqrt{p_{x,\mathrm{tot}}^2 + p_{y,\mathrm{tot}}^2}$.
Add one more `Define` for `pt` and histogram it (try a range 0–5 GeV).

In [ ]:
# TODO: define "pt" from px_tot and py_tot, then histogram it
# reco_pt = reco.Define("pt", "sqrt(px_tot*px_tot + py_tot*py_tot)")
# h_pt = reco_pt.Histo1D(("h_pt", "p_{T}; p_{T} [GeV]; events", 60, 0, 5), "pt")
# c = ROOT.TCanvas(); h_pt.Draw(); c.Draw()

## 4. Detector resolution: smear the momenta

A real detector never measures momentum perfectly. We model that by multiplying
each daughter's $p_x, p_y$ by a random factor $(1 + \sigma\cdot\mathcal{N}(0,1))$.

The little C++ helper below does the smearing. **You don't need to read it** — we just
declare it to ROOT once so RDataFrame can use it by name.

In [ ]:
ROOT.gInterpreter.Declare(r'''
#include <ROOT/RVec.hxx>
#include "TRandom.h"
// toy momentum smearing: scale each component by (1 + sigma * Gaussian)
ROOT::RVec<float> smear(const ROOT::RVec<float>& v, double sigma){
    ROOT::RVec<float> out(v.size());
    for (std::size_t i = 0; i < v.size(); ++i)
        out[i] = v[i] * (1.0 + sigma * gRandom->Gaus());
    return out;
}
''')
ROOT.gRandom.SetSeed(123)   # reproducible

Now a helper that builds the smeared invariant-mass histogram for a given $\sigma$:

In [ ]:
def smeared_mass_hist(sigma, name):
    d = (df
        .Filter("d_px.size() == 2")
        .Define("px_s", f"smear(d_px, {sigma})")
        .Define("py_s", f"smear(d_py, {sigma})")
        .Define("E_s",  "sqrt(px_s*px_s + py_s*py_s + d_pz*d_pz + d_m*d_m)")
        .Define("m_s",  "sqrt(pow(Sum(E_s),2) - pow(Sum(px_s),2) - pow(Sum(py_s),2) - pow(Sum(d_pz),2))")
    )
    return d.Histo1D((name, "mass vs smearing; m_{inv} [GeV]; events", 60, 0.6, 1.4), "m_s")

## 5. Overlay several resolutions

Draw the reconstructed mass for $\sigma = 0,\,1\%,\,5\%,\,10\%$ on the same axes.
Watch the peak broaden as the detector gets worse.

In [ ]:
sigmas = [0.00, 0.01, 0.05, 0.10]
colors = [ROOT.kCyan+1, ROOT.kBlue, ROOT.kViolet, ROOT.kMagenta]

c2 = ROOT.TCanvas("c2", "", 800, 600)
leg = ROOT.TLegend(0.62, 0.66, 0.88, 0.88)
leg.SetBorderSize(0)

hists = []   # keep references alive
for i, (s, col) in enumerate(zip(sigmas, colors)):
    h = smeared_mass_hist(s, f"h_{int(s*100)}").GetValue()
    h.SetLineColor(col); h.SetLineWidth(2); h.SetDirectory(0)
    hists.append(h)
    h.Draw("hist" if i == 0 else "hist same")
    leg.AddEntry(h, f"#sigma = {int(s*100)}%", "l")

leg.Draw()
c2.Draw()

## 6. TODO — quantify the resolution

The peak gets wider with $\sigma$. Put a number on it: for each $\sigma$, print the
**mass resolution** = the standard deviation of the smeared mass.

*Hint:* build the same dataframe as in `smeared_mass_hist`, but return `d.StdDev("m_s")`
instead of a histogram.

In [ ]:
# TODO: for each sigma in [0.0, 0.01, 0.05, 0.10], print StdDev of the smeared mass
# for s in [0.0, 0.01, 0.05, 0.10]:
#     d = (df.Filter("d_px.size() == 2")
#            .Define("px_s", f"smear(d_px, {s})")
#            ... )
#     print(f"sigma={s:>4}:  mass resolution = {d.StdDev('m_s').GetValue():.4f} GeV")

## 7. What did you find?

- The **mean** mass stays at ~1 GeV — the smearing is unbiased on average.
- The **width** grows with $\sigma$ — that's your detector resolution propagating into
  the measurement. A wider peak makes a signal harder to separate from background.

This is exactly the trade-off a real experiment optimises.

## 8. (optional) The full study

The complete analysis also reconstructs the **decay vertex** from the two daughter
tracks (the DOCA method) and measures how the *vertex* resolution degrades with
$\sigma$. It's in `reference/llp_simple_analysis.py` (and the C++ `.C`), with a
detailed write-up in `reference/README.md`. Have a look if you're curious.

## 9. Save your work with git

You changed this notebook — snapshot it so it's safe:

In [ ]:
!git add tutorial.ipynb
!git commit -m "Complete pT and resolution TODOs"